# FRED Metadata Collection

How the FRED API is used to build the series catalogue in Postgres, and why it is built
this way. The code is `apps/agentic/db/fred_discovery.py`; the entry point is
`clients/bin/fred_metadata.py`; the specification is
`sefer/planning/yada-fred-subcategory-procedure.md`.

**This notebook is safe to re-run.** Every write below is idempotent — series are
upserted, memberships inserted on-conflict-do-nothing, and a category whose scan is
already `ok` is skipped entirely. Section 7 demonstrates that rather than asserting it.

### Why there is a collection procedure at all

The catalogue used to come from YAML exports under `clients/fred/`, read back by a
loader. That export was wrong in four ways and reported none of them:

| defect | size |
| --- | --- |
| series lists truncated at one 1,000-row page | 145,192 missing memberships |
| tree rooted a level too low | 7 of 12 "top-level" names were second-level |
| `Production & Business Activity` absent | 3,274 series + 16 children |
| tree incomplete | 881 of 1,058 non-regional categories |

A file has no transaction, no row count and nothing to reconcile against, so one page of
nine looks exactly like completion. **Hence the rule this whole module is built on: every
fetch stores what FRED said beside what was actually written.**

## 0. Setup

In [1]:
import os
import sys
import json

sys.path.insert(0, os.path.abspath('../..'))

from dotenv import load_dotenv
load_dotenv(os.path.abspath('../../.env'), override=True)

from sqlalchemy import text

from apps.agentic.db.fred_catalogue import _engine
from apps.agentic.db import fred_discovery as fd

engine = _engine()
print('alembic head in db:',
      engine.connect().execute(text('SELECT version_num FROM alembic_version')).scalar())

alembic head in db: 0009


## 1. The paced client

`FredApi` is a thin paced GET. Three things in it are not incidental:

- **`MIN_INTERVAL = 0.6`** — about 100 requests/minute against FRED's documented ceiling
  of 120. Probing at ~480/min returned `429`s and killed a run.
- **429 and 5xx back off and retry**; other 4xx raise immediately.
- **A `400` mentioning `api_key` raises `FredApiError` rather than retrying.** yada's key
  was unregistered for an unknown period and nothing noticed, because nothing in yada
  called the API. A dead key must fail loudly, not look like rate limiting.

In [2]:
print('page size      :', fd.PAGE, '  (FRED maximum for category/series)')
print('min interval   :', fd.MIN_INTERVAL, 's ->', round(60 / fd.MIN_INTERVAL), 'req/min')
print('max walk depth :', fd.MAX_DEPTH, ' (a cycle guard, not a FRED limit)')

api = fd.FredApi()          # reads FRED_API_KEY; raises if absent
root = api.get('category/children', category_id=fd.ROOT_CATEGORY)
for c in root['categories']:
    print(f"  [{c['id']:>6}] {c['name']}")

page size      : 1000   (FRED maximum for category/series)
min interval   : 0.6 s -> 100 req/min
max walk depth : 12  (a cycle guard, not a FRED limit)


  [ 32991] Money, Banking, & Finance
  [    10] Population, Employment, & Labor Markets
  [ 32992] National Accounts
  [     1] Production & Business Activity
  [ 32455] Prices
  [ 32263] International Data
  [  3008] U.S. Regional Data
  [ 33060] Academic Data


In [3]:
# A bad key fails loudly instead of being retried as a rate limit.
try:
    fd.FredApi(api_key='0' * 32).get('category', category_id=0)
except fd.FredApiError as error:
    print('raised as designed:', str(error)[:100])

raised as designed: FRED rejected the API key: {"error_code":400,"error_message":"Bad Request.  The value for variable a


## 2. Walking the category tree

Breadth-first from category `0`, taking **`parent_id` from FRED's own response**.

The old export kept only a path string, so a subtree fetched by its own id lost the
levels above it with nothing able to detect the loss — `Countries` was really
`International Data > Countries`. Storing the edge and *deriving* the path means the two
cannot disagree.

In [4]:
page = api.get('category/children', category_id=27281)   # U.S. Regional Data > States
print(f"{len(page['categories'])} children; FRED supplies parent_id directly:")
for c in page['categories'][:4]:
    print(f"  id={c['id']:<7} parent_id={c.get('parent_id')}  {c['name']}")

53 children; FRED supplies parent_id directly:
  id=27282   parent_id=27281  Alabama
  id=27283   parent_id=27281  Alaska
  id=27284   parent_id=27281  Arizona
  id=149     parent_id=27281  Arkansas


In [5]:
# The path is derived from parent_id with a recursive CTE, never stored independently.
with engine.connect() as c:
    for row in c.execute(text('''
        SELECT category_id, parent_id, walked_at IS NOT NULL AS walked, category_path
          FROM fred_category
         WHERE category_id IN (3008, 27281, 27282, 32267, 33062)
         ORDER BY category_path
    ''')):
        print(f'  [{row[0]:>6}] parent={str(row[1]):<7} walked={row[2]!s:<5} {row[3]}')

  [ 33062] parent=33061   walked=True  Academic Data > NBER Macrohistory Database > Production of Commodities
  [ 32267] parent=32264   walked=True  International Data > Countries > United States
  [  3008] parent=None    walked=True  U.S. Regional Data
  [ 27281] parent=3008    walked=True  U.S. Regional Data > States
  [ 27282] parent=27281   walked=True  U.S. Regional Data > States > Alabama


`walked_at` is what makes the walk resumable. A category is **found** when its parent
names it, and **visited** when we ask it for children — different things. Recording only
the first meant a restart re-asked thousands of categories for children they had already
reported, and left no way to tell a real leaf from a node never reached.

A resuming walk skips visited categories and queues their children *from the database*,
so it continues past the previous frontier instead of stopping at it. A **refresh** must
pass `revisit=True` — skipping would never notice categories FRED has added.

In [6]:
with engine.connect() as c:
    walked = c.execute(text('SELECT count(*) FROM fred_category WHERE walked_at IS NOT NULL')).scalar()
    total = c.execute(text('SELECT count(*) FROM fred_category')).scalar()
print(f'visited {walked:,} of {total:,} categories; {total - walked:,} still to walk')

visited 5,201 of 5,201 categories; 0 still to walk


## 3. Paginated fetch, and the reconciliation

**This is the section that matters.** `category/series` returns at most 1,000 rows. The
old export took the first page and stopped, which is how 80 categories came to hold
exactly 1,000 series each — a page size recorded as a count.

Two details in the loop:

- `offset` advances by **the page actually returned**, not by the requested limit.
  Advancing by the limit skips a window whenever the server returns a short page.
- the loop ends at `offset >= count`, so an exact multiple costs no wasted trailing call.

In [7]:
DEMO = 33062   # Academic Data > NBER Macrohistory Database > Production of Commodities

pages = list(fd.iter_category_series(api, DEMO))
for n, (rows, count) in enumerate(pages, 1):
    print(f'  page {n}: {len(rows):>5} series, FRED reports count={count:,}')
print(f'\n  pages fetched: {len(pages)}, series seen: {sum(len(r) for r, _ in pages):,}')

  page 1:   454 series, FRED reports count=454

  pages fetched: 1, series seen: 454


The pair of numbers is the whole contract. `fetch_category` writes both:

> `api_series_count > fetched_series_count` **with** a `page_limit` is a choice.
> The same inequality **without** one is a fault.

`status` is `ok` only when we wrote what we asked for. Nothing else is trusted.

In [8]:
outcome = fd.fetch_category(engine, api, DEMO)   # idempotent: safe to re-run
print(json.dumps(outcome, indent=2))

{
  "category_id": 33062,
  "api_series_count": 454,
  "fetched_series_count": 454,
  "pages": 1,
  "status": "ok",
  "page_limit": null,
  "capped": false
}


In [9]:
with engine.connect() as c:
    for row in c.execute(text('''
        SELECT status, count(*), sum(api_series_count), sum(fetched_series_count),
               count(*) FILTER (WHERE page_limit IS NOT NULL) AS capped
          FROM fred_category_scan GROUP BY status ORDER BY 2 DESC
    ''')):
        print(f'  {row[0]:<11} {row[1]:>5} categories  '
              f'api={row[2] or 0:>9,}  written={row[3] or 0:>9,}  capped={row[4]}')

  ok           5201 categories  api=1,052,462  written=1,000,709  capped=245


## 4. Deliberate limits, recorded as data

Country and state data were originally left out **on purpose**, because of their volume.
Nothing recorded that, so a later reader diagnosed both as faults. An unrecorded decision
is indistinguishable from a bug — the same failure as a silent short fetch, one level up.

- **`fred_category_scan.page_limit`** — a deliberate page ceiling. Countries stay at one
  page. Without this column a capped country reads `status='incomplete'`: alarming, wrong,
  and retried on every resume forever.
- **`fred_category.excluded_reason`** — a subtree recorded but not fetched. The categories
  stay in the tree, so a browse can say "this exists and was excluded" instead of showing
  an unexplained gap, and including it later needs no re-walk.

`mark_exclusions` reconciles **both ways**: emptying `VOLUME_EXCLUDED` clears the stored
marks. Otherwise an exclusion outlives its decision and the fetch skips a branch that is
wanted again.

**Both halves of that have now been exercised.** The U.S. Regional subtree was excluded on
2026-11-01 and collected on 2026-11-07 — emptying the table, letting the reconciler clear
4,130 stored marks, and fetching the branch without re-walking it. The country cap is still
in force, because BIS is the better source for country data. So `VOLUME_EXCLUDED` prints
empty below while `page_limit` is still set on 245 categories: one decision reversed, one
standing, both legible.

In [10]:
print('VOLUME_EXCLUDED =', fd.VOLUME_EXCLUDED or '{}  (nothing excluded)')
print('country page cap:', fd.page_limit_for('International Data > Countries > Belgium'))
print('elsewhere       :', fd.page_limit_for('Prices > Commodities'))

with engine.connect() as c:
    row = c.execute(text('''
        SELECT count(*), sum(api_series_count), sum(fetched_series_count)
          FROM fred_category_scan WHERE page_limit IS NOT NULL
    ''')).fetchone()
print(f'\ncapped categories: {row[0]}')
print(f'  FRED has {row[1] or 0:,}, we hold {row[2] or 0:,}, '
      f'deferred {(row[1] or 0) - (row[2] or 0):,} -- visible, not missing')

VOLUME_EXCLUDED = {}  (nothing excluded)
country page cap: 1
elsewhere       : None

capped categories: 245
  FRED has 137,413, we hold 85,660, deferred 51,753 -- visible, not missing


## 5. The two derived columns

Neither can be reconstructed by a query, so both are computed at load.

**`frequency_class`** — the source spells exactly eight frequencies **46 ways**; annual alone
is spelled 12 ways. `frequency = 'Quarterly'` silently drops the series spelled
*"Quarterly, End of Quarter"*.

**`currently_relevant`** — the highest-value filter available, and a flat date cutoff gets it
catastrophically wrong. Annual is **59% of the catalogue** and an annual series reporting last
year is exactly on schedule, so a flat `observation_end >= '2026-01-01'` keeps just **1,523 of
498,774 annual series (0.3%)** while the per-class rule keeps **367,404 (73.7%)**. Across the
whole catalogue the flat cutoff discards 56% of what the period rule keeps.

So currency is measured in **periods**: the cutoff is a per-class p95 of `observation_end`
minus two periods. p95 rather than max, because projections skew the tail; two periods, one
for publication lag and one for the catalogue's own staleness. It is *stored*, so a tool can
state the definition it applied instead of hiding it, and it is re-derived on every load —
which is why it needed no adjustment when the catalogue doubled.

In [11]:
print('frequency_class collapses raw spellings:')
with engine.connect() as c:
    for row in c.execute(text('''
        SELECT frequency_class, count(DISTINCT frequency) spellings, count(*) series
          FROM fred_series GROUP BY frequency_class ORDER BY series DESC
    ''')):
        print(f'  {row[0]:<12} {row[1]:>2} spellings  {row[2]:>8,} series')

frequency_class collapses raw spellings:


  annual       12 spellings   498,774 series
  monthly       8 spellings   220,789 series
  quarterly     6 spellings   108,960 series
  daily         3 spellings    11,478 series
  weekly        9 spellings     3,631 series
  semiannual    2 spellings     2,113 series
  other         3 spellings       867 series
  biweekly      3 spellings        16 series


In [12]:
with engine.connect() as c:
    print('the stored currency rule:')
    for row in c.execute(text('''
        SELECT frequency_class, period_days, p95_observation_end, currency_cutoff, series_count
          FROM fred_frequency_class ORDER BY series_count DESC
    ''')):
        print(f'  {row[0]:<12} period={row[1]:>5}d  p95={row[2]}  cutoff={row[3]}  n={row[4]:,}')

the stored currency rule:
  annual       period=  365d  p95=2025-01-01  cutoff=2023-01-02  n=498,774
  monthly      period=   30d  p95=2026-10-01  cutoff=2026-08-02  n=220,789
  quarterly    period=   91d  p95=2026-07-01  cutoff=2025-12-31  n=108,960
  daily        period=    1d  p95=2026-10-29  cutoff=2026-10-27  n=11,478
  weekly       period=    7d  p95=2026-11-04  cutoff=2026-10-21  n=3,631
  semiannual   period=  182d  p95=2026-01-01  cutoff=2025-01-02  n=2,113
  other        period=  365d  p95=2026-09-16  cutoff=2024-09-16  n=867
  biweekly     period=   14d  p95=2026-10-29  cutoff=2026-10-01  n=16


## 6. Coverage

The denominator has to come from somewhere the category walk cannot influence, or it just
restates what we fetched. Releases provide that: each series belongs to one release, so
enumerating every release's series list gives an independent total.

Two ways to ask, and they are not equivalent:

- **Sum the per-release counts** — 337 requests, one per release with `limit=1`. Cheap, but
  it over-counts by however many series belong to more than one release.
- **Union the per-release series ids** — 1,132 requests, every page of every release. Exact.

Measured 2026-11-07, both were **853,320**, with **zero series in two or more releases**. That
agreement is the useful part: it turns one-release-per-series from a documented claim into a
measured property, so the cheap sum is a safe denominator from here on.

Neither sweep runs below — it would cost over a thousand requests to reproduce a constant. The
figures are recorded as constants with the date they were taken.

Read-only below.

In [13]:
# Measured 2026-11-07 by enumerating all 337 releases (1,132 requests).
# The union and the sum of per-release counts agreed exactly, and no series appeared in
# two releases -- so this is a true total, not an over-count.
FRED_TOTAL = 853_320
RELEASE_LISTED = 845_628    # held AND listed by some release
NO_RELEASE = 1_000          # held but listed by NO release -- see the note below

with engine.connect() as c:
    held = c.execute(text('SELECT count(*) FROM fred_series')).scalar()
    cats = c.execute(text('SELECT count(*) FROM fred_category')).scalar()
    excl = c.execute(text('SELECT count(*) FROM fred_category WHERE excluded_reason IS NOT NULL')).scalar()
    notes = c.execute(text('SELECT count(*) FROM fred_series WHERE notes IS NOT NULL')).scalar()
    cur = c.execute(text('SELECT count(*) FROM fred_series WHERE currently_relevant')).scalar()
    deferred = c.execute(text('''SELECT COALESCE(sum(api_series_count - fetched_series_count), 0)
        FROM fred_category_scan WHERE page_limit IS NOT NULL''')).scalar()

print(f'FRED total series   : {FRED_TOTAL:>9,}')
print(f'held                : {held:>9,}')
print(f'  listed by release : {RELEASE_LISTED:>9,}   ({100 * RELEASE_LISTED / FRED_TOTAL:.1f}% coverage)')
print(f'  listed by none    : {NO_RELEASE:>9,}   <- exactly PAGE; see below')
print(f'missing             : {FRED_TOTAL - RELEASE_LISTED:>9,}   '
      f'(7,690 of them behind the country cap)')
print()
print(f'deferred by the cap : {deferred:>9,}   memberships, recorded not lost')
print(f'categories          : {cats:>9,}  ({excl:,} excluded)')
print(f'with notes          : {notes:>9,}   ({100 * notes / max(held, 1):.1f}%)')
print(f'currently relevant  : {cur:>9,}   ({100 * cur / max(held, 1):.1f}%)')

FRED total series   :   853,320
held                :   846,628
  listed by release :   845,628   (99.1% coverage)
  listed by none    :     1,000   <- exactly PAGE; see below
missing             :     7,692   (7,690 of them behind the country cap)

deferred by the cap :    51,753   memberships, recorded not lost
categories          :     5,201  (0 excluded)
with notes          :   774,931   (91.5%)
currently relevant  :   585,251   (69.1%)


**The 1,000 is a finding, and it was traced.** 1,000 is exactly `PAGE`, which is what made it
worth chasing rather than rounding off. All 1,000 sit in one category — 34111, *Senior Credit
Officer Opinion Survey*, holding 2,398 memberships against FRED's reported 1,398 — and every
one belongs to the retired `ALLQ*` series family. FRED returns `400 Bad Request` for each, none
carry `notes` while all 1,398 current rows do, and the live ids use entirely different prefixes.

FRED replaced the family; the superseded YAML loader had written exactly 1,000 of them, its
truncation limit; and **nothing in the fetch path deletes**, so they stayed. That last part is
deliberate — a partial run must leave the catalogue larger, never emptier.

**What it exposed is a blind spot in the reconciliation, not a failure of it.** The scan compares
`api_series_count` against `fetched_series_count` — what FRED reported against what *that run
wrote*. That proves a fetch was complete and is structurally incapable of seeing rows that were
already present, which is why `scans_bad` is 0 and the sums close exactly while the category
holds 1,000 retired series. The missing half of the check is `held > api_series_count`:

In [14]:
# The other direction of the reconciliation. Reporting only -- pruning needs a complete
# verified scan to be safe, since a failed or capped fetch would look identical.
surplus = fd.surplus_categories(engine)
if not surplus:
    print('no category holds more memberships than FRED reports')
for category_id, path, api_count, held in surplus:
    print(f'category {category_id}: FRED reports {api_count:,}, we hold {held:,} '
          f'(+{held - api_count:,})')
    print(f'  {path}')

category 34111: FRED reports 1,398, we hold 2,398 (+1,000)
  Money, Banking, & Finance > Banking > Senior Credit Officer Opinion Survey


## 7. Re-run safety

The claim is that running this notebook again changes nothing. Demonstrated, not asserted:
fetch the same category twice and compare row counts.

In [15]:
def snapshot():
    with engine.connect() as c:
        return {
            'series': c.execute(text('SELECT count(*) FROM fred_series')).scalar(),
            'memberships': c.execute(text('SELECT count(*) FROM fred_series_category')).scalar(),
        }

before = snapshot()
fd.fetch_category(engine, api, DEMO)
after = snapshot()
print('before:', before)
print('after :', after)
print('unchanged:', before == after)

before: {'series': 846628, 'memberships': 1002244}
after : {'series': 846628, 'memberships': 1002244}
unchanged: True


In [16]:
# The same property at the level of the whole pipeline: a category already `ok` is not
# re-fetched at all, so a second collection run costs no requests.
pending = fd.pending_categories(engine)
print(f'categories still pending a fetch: {len(pending):,}')
print('(0 means the catalogue is complete and `fred_metadata.py all` is a no-op)')

categories still pending a fetch: 0
(0 means the catalogue is complete and `fred_metadata.py all` is a no-op)


In [17]:
# Reconciling exclusions is also convergent: the second call reports nothing to do.
print('reconcile:', fd.mark_exclusions(engine))
print('reconcile:', fd.mark_exclusions(engine))

reconcile: {'marked': 0, 'cleared': 0}
reconcile: {'marked': 0, 'cleared': 0}


## 8. Running it outside a notebook

```bash
python clients/bin/fred_metadata.py status            # read-only
python clients/bin/fred_metadata.py all --dry-run     # what would happen
python clients/bin/fred_metadata.py all               # walk, then fetch
python clients/bin/fred_metadata.py fetch --limit 5   # bound a trial run
python clients/bin/fred_metadata.py walk --revisit    # refresh, not resume
python clients/bin/fred_metadata.py resync            # after FRED changes
```

A full collection from empty is **about 10,700 requests and roughly 75 minutes** at the
achievable rate. Measured: the non-regional half took 2,359 requests / 28 minutes, the
regional half 8,342 requests / 45 minutes, sustaining 94 requests per minute with no
failures and nothing incomplete.

It is interruptible at any point: nothing in the module deletes, so a partial run leaves
the catalogue larger and more correct, never emptier. A resumed run re-asks nothing — the
regional walk skipped 1,058 already-visited categories without spending a request.

**Why a script and not only this notebook.** Convention here is that a slow or throttled
download gets a notebook, because the saved cell output is the record that an expensive
pull succeeded. FRED has both, and the reason is that the record lives in
`fred_category_scan` instead: every category's completeness is a queryable row, so the
run does not need a human watching it. A notebook cannot resume, and a 45-minute paced
pull that dies at 40 needs to.